# Imports

In [ ]:
YOUR_USERNAME = ""
YOUR_TIMESTAMP = ""

In [ ]:
import sys, os
from pathlib import Path

sys.path.append(f"/kaggle/input/datasets/{YOUR_USERNAME}/nfl-src-2026")

In [ ]:
import json
import numpy as np
import joblib

from src.config import Config
from src.preprocess import load_and_preprocess
from src.model import train_model
from src.utils import (
    set_seed,
    set_folder,
)

# Data

In [ ]:
Config.DATA_DIR = Path("/kaggle/input/competitions/nfl-big-data-bowl-2026-prediction")
Config.TRAIN_PATH = Config.DATA_DIR / "train"
Config.OUTPUT_DIR = Path(f"/kaggle/input/datasets/{YOUR_USERNAME}/nfl-weights-2026")
Config.G_DATA_DIR = Path(f"/kaggle/input/datasets/{YOUR_USERNAME}/nfl-scaler-2026")
Config.G_DATA_VERSION = 25
Config.DEBUG = False
Config.TRAIN = False

In [ ]:
import joblib
from src.config import Config

scaler = joblib.load(Config.G_DATA_DIR / f"scaler_25.pkl")
print("Scaler expects:", scaler.n_features_in_)
print("Config.feature_cols length:", len(Config.feature_cols))

# Submission

In [ ]:
Config.TIME_TAG = YOUR_TIMESTAMP
SEEDS = [888, 3407, 0, 1, 42]

In [ ]:
"""
The evaluation API requires that you set up a server which will respond to inference requests.
We have already defined the server; you just need write the predict function.
When we evaluate your submission on the hidden test set the client defined in `nfl_gateway` will run in a different container
with direct access to the hidden test set and hand off the data timestep by timestep.
Your code will always have access to the published copies of the copmetition files.
"""

import os

import pandas as pd
import polars as pl

import kaggle_evaluation.nfl_inference_server
from src.predict import (
    pf_inference,
)

if Config.TRAIN:
    Config.TRAIN = False


def predict(
    test: pl.DataFrame, test_input: pl.DataFrame
) -> pl.DataFrame | pd.DataFrame:
    """Replace this function with your inference code.
    You can return either a Pandas or Polars dataframe, though Polars is recommended for performance.
    Each batch of predictions (except the very first) must be returned within 5 minutes of the batch features being provided.
    """
    # predictions = pl.DataFrame({'x': [0.0] * len(test), 'y': [0.0] * len(test)})
    if Config.TRAIN:
        Config.TRAIN = False

    if isinstance(test, pl.DataFrame):
        test = test.to_pandas()
    if isinstance(test_input, pl.DataFrame):
        test_input = test_input.to_pandas()

    predictions = pf_inference(test, test_input, time_tag=Config.TIME_TAG, seeds=SEEDS)

    assert isinstance(predictions, (pd.DataFrame, pl.DataFrame))
    assert len(predictions) == len(test)
    return predictions


# When your notebook is run on the hidden test set, inference_server.serve must be called within 10 minutes of the notebook starting
# or the gateway will throw an error. If you need more than 15 minutes to load your model you can do so during the very
# first `predict` call, which does not have the usual 5 minute response deadline.
inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    inference_server.run_local_gateway((Config.DATA_DIR,))